<a href="https://colab.research.google.com/github/srijabiswas-01/Data_Analysis_and_Visualization/blob/main/E_Commerce_Dashboard_making_using_Ploty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dash plotly pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 73.0 MB/s eta 0:00:00


In [ ]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download(
    "abbas829/e-commerce-sales-analytics-dataset"
)

print("Downloaded to:", path)

files = os.listdir(path)
print(files)

100%|██████████| 108k/108k [00:00<00:00, 63.6MB/s]

Extracting files...
Downloaded to: /root/.cache/kagglehub/datasets/abbas829/e-commerce-sales-analytics-dataset/versions/1
['ecommerce_sales_analytics_5000.csv']


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

from dash import Dash
from dash import dcc
from dash import html
from dash import Input, Output

import plotly.express as px

csv_file = os.path.join(
    path,
    "ecommerce_sales_analytics_5000.csv"
)

df = pd.read_csv(csv_file)

df['order_date'] = pd.to_datetime(df['order_date'])

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month_name()
df['month_num'] = df['order_date'].dt.month
df['quarter'] = df['order_date'].dt.quarter
df['day_name'] = df['order_date'].dt.day_name()

df.head()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue,year,month,month_num,quarter,day_name
0,10001,2022-01-01,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20,2022,January,1,1,Saturday
1,10002,2022-01-02,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10,2022,January,1,1,Sunday
2,10003,2022-01-03,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35,2022,January,1,1,Monday
3,10004,2022-01-04,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90,2022,January,1,1,Tuesday
4,10005,2022-01-05,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56,2022,January,1,1,Wednesday


In [ ]:
df['gross_sales'] = df['quantity'] * df['unit_price']

df['discount_amount'] = (
    df['gross_sales'] * df['discount']
)

df['net_revenue'] = (
    df['gross_sales'] - df['discount_amount']
)

df['delivery_performance'] = np.where(
    df['delivery_days'] <= 5,
    'Fast',
    'Slow'
)

df['rating_segment'] = pd.cut(
    df['customer_rating'],
    bins=[0,2,3,4,5],
    labels=[
        'Poor',
        'Average',
        'Good',
        'Excellent'
    ]
)


In [ ]:
TOTAL_REVENUE = df['revenue'].sum()

TOTAL_ORDERS = df['order_id'].nunique()

TOTAL_CUSTOMERS = df['customer_id'].nunique()

AVG_ORDER_VALUE = df['revenue'].mean()

AVG_RATING = df['customer_rating'].mean()

TOTAL_DISCOUNT_GIVEN = df['discount_amount'].sum()

AVG_DELIVERY_DAYS = df['delivery_days'].mean()

print("="*50)
print("BUSINESS KPI SUMMARY")
print("="*50)

print(f"Total Revenue: ${TOTAL_REVENUE:,.2f}")
print(f"Total Orders: {TOTAL_ORDERS:,}")
print(f"Total Customers: {TOTAL_CUSTOMERS:,}")
print(f"Average Order Value: ${AVG_ORDER_VALUE:,.2f}")
print(f"Average Customer Rating: {AVG_RATING:.2f}")
print(f"Total Discounts Given: ${TOTAL_DISCOUNT_GIVEN:,.2f}")
print(f"Average Delivery Days: {AVG_DELIVERY_DAYS:.2f}")

BUSINESS KPI SUMMARY
Total Revenue: $5,109,775.74
Total Orders: 5,000
Total Customers: 989
Average Order Value: $1,021.96
Average Customer Rating: 2.97
Total Discounts Given: $1,130,246.08
Average Delivery Days: 6.12


In [ ]:
category_sales = (
    df.groupby('product_category')
      .agg(
          Revenue=('revenue','sum'),
          Orders=('order_id','count'),
          Avg_Rating=('customer_rating','mean')
      )
      .reset_index()
      .sort_values('Revenue',ascending=False)
)

fig = px.bar(
    category_sales,
    x='product_category',
    y='Revenue',
    color='Revenue',
    title='Revenue by Product Category'
)

fig.show()


In [ ]:
region_sales = (
    df.groupby('region')
      .agg(
          Revenue=('revenue','sum'),
          Orders=('order_id','count')
      )
      .reset_index()
)

fig = px.pie(
    region_sales,
    names='region',
    values='Revenue',
    hole=0.45,
    title='Revenue Distribution by Region'
)

fig.show()

In [ ]:
monthly_sales = (
    df.groupby(
        pd.Grouper(
            key='order_date',
            freq='M'
        )
    )['revenue']
    .sum()
    .reset_index()
)

fig = px.line(
    monthly_sales,
    x='order_date',
    y='revenue',
    markers=True,
    title='Monthly Revenue Trend'
)

fig.show()

/tmp/ipykernel_2527/1531412534.py:3: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [ ]:
payment_analysis = (
    df.groupby('payment_method')
      .agg(
          Revenue=('revenue','sum'),
          Orders=('order_id','count')
      )
      .reset_index()
)

fig = px.bar(
    payment_analysis,
    x='payment_method',
    y='Revenue',
    color='payment_method',
    title='Revenue by Payment Method'
)

fig.show()

In [ ]:
fig = px.histogram(
    df,
    x='customer_rating',
    nbins=20,
    title='Customer Rating Distribution'
)

fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='discount',
    y='revenue',
    color='product_category',
    size='quantity',
    hover_data=['region'],
    title='Discount vs Revenue'
)

fig.show()


In [ ]:
delivery_region = (
    df.groupby('region')
      .agg(
          Avg_Delivery=('delivery_days','mean')
      )
      .reset_index()
)

fig = px.bar(
    delivery_region,
    x='region',
    y='Avg_Delivery',
    color='region',
    title='Average Delivery Days by Region'
)

fig.show()

In [ ]:
top_customers = (
    df.groupby('customer_id')
      .agg(
          Revenue=('revenue','sum')
      )
      .reset_index()
      .sort_values('Revenue',ascending=False)
      .head(20)
)

fig = px.bar(
    top_customers,
    x='customer_id',
    y='Revenue',
    title='Top 20 Customers by Revenue'
)

fig.show()

In [ ]:
corr_cols = [
    'quantity',
    'unit_price',
    'discount',
    'delivery_days',
    'customer_rating',
    'revenue'
]

corr = df[corr_cols].corr()

fig = px.imshow(
    corr,
    text_auto=True,
    aspect='auto',
    title='Correlation Matrix'
)

fig.show()

In [ ]:
category_rating = (
    df.groupby('product_category')
      .agg(
          Avg_Rating=('customer_rating','mean')
      )
      .reset_index()
)

fig = px.bar(
    category_rating,
    x='product_category',
    y='Avg_Rating',
    color='Avg_Rating',
    title='Average Rating by Category'
)

fig.show()

In [ ]:
revenue_per_customer = TOTAL_REVENUE / TOTAL_CUSTOMERS

kpi_titles = [
    "Revenue",
    "Orders",
    "Customers",
    "Avg Order",
    "Rating",
    "Delivery Days",
    "Discounts",
    "Revenue/Customer"
]

kpi_values = [
    f"${TOTAL_REVENUE:,.0f}",
    f"{TOTAL_ORDERS:,}",
    f"{TOTAL_CUSTOMERS:,}",
    f"${AVG_ORDER_VALUE:,.0f}",
    f"{AVG_RATING:.2f}",
    f"{AVG_DELIVERY_DAYS:.2f}",
    f"${TOTAL_DISCOUNT_GIVEN:,.0f}",
    f"${revenue_per_customer:,.0f}"
]

In [ ]:
fig = make_subplots(
    rows=5,
    cols=2,

    specs=[
        [{"type":"xy"},{"type":"xy"}],
        [{"type":"domain"},{"type":"xy"}],
        [{"type":"xy"},{"type":"xy"}],
        [{"type":"xy"},{"type":"xy"}],
        [{"type":"xy"},{"type":"heatmap"}]
    ],

    subplot_titles=(
        "Monthly Revenue Trend",
        "Revenue by Category",

        "Revenue by Region",
        "Revenue by Payment Method",

        "Discount vs Revenue",
        "Customer Ratings",

        "Top Customers",
        "Delivery Days by Region",

        "Category Ratings",
        "Correlation Matrix"
    ),

    vertical_spacing=0.08
)
fig.add_trace(
    go.Scatter(
        x=monthly_sales['order_date'],
        y=monthly_sales['revenue'],
        mode='lines+markers',
        name='Revenue Trend'
    ),
    row=1,col=1
)

fig.add_trace(
    go.Bar(
        x=category_sales['product_category'],
        y=category_sales['Revenue'],
        name='Category Revenue'
    ),
    row=1,col=2
)
fig.add_trace(
    go.Pie(
        labels=region_sales['region'],
        values=region_sales['Revenue'],
        hole=.5
    ),
    row=2,col=1
)

fig.add_trace(
    go.Bar(
        x=payment_analysis['payment_method'],
        y=payment_analysis['Revenue']
    ),
    row=2,col=2
)
fig.add_trace(
    go.Scatter(
        x=df['discount'],
        y=df['revenue'],
        mode='markers',
        marker=dict(size=6)
    ),
    row=3,col=1
)

fig.add_trace(
    go.Histogram(
        x=df['customer_rating']
    ),
    row=3,col=2
)
fig.add_trace(
    go.Bar(
        x=top_customers['customer_id'],
        y=top_customers['Revenue']
    ),
    row=4,col=1
)

fig.add_trace(
    go.Bar(
        x=delivery_region['region'],
        y=delivery_region['Avg_Delivery']
    ),
    row=4,col=2
)
fig.add_trace(
    go.Bar(
        x=category_rating['product_category'],
        y=category_rating['Avg_Rating']
    ),
    row=5,col=1
)

fig.add_trace(
    go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.columns
    ),
    row=5,col=2
)
fig.update_layout(

    title={
        "text":"E-Commerce Sales Analytics Executive Dashboard",
        "x":0.5
    },

    height=2200,
    width=1600,

    template="plotly_white",

    showlegend=False
)

fig.show()